## TSP

In [ ]:
%pip install pulp pandas -q

import pulp
import pandas as pd
import math

# DADOS

# Coordenadas (para calcular distancias)
coords = {
    0: (0, 0),
    1: (3, 0),
    2: (5, 4),
    3: (2, 5),
    4: (6, 3) # 5 cidades
}
n = len(coords)
cidades = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

# Matriz de distancias (arredondada para int)
dist = [[0]*n for _ in range(n)]
for i in range(n):
    for j in range(n):
        if i != j:
            dx = coords[i][0] - coords[j][0]
            dy = coords[i][1] - coords[j][1]
            dist[i][j] = int(round(math.sqrt(dx*dx + dy*dy)))
        else:
            dist[i][j] = 0

# MODELO PLI (MTZ)

prob = pulp.LpProblem("TSP", pulp.LpMinimize)

# Variaveis x_ij binarias (se o arco i->j é usado)
x = [[pulp.LpVariable(f"x_{i}_{j}", cat='Binary') for j in range(n)] for i in range(n)]

# Variaveis u_i (ordem de visita, contínuas)
u = [pulp.LpVariable(f"u_{i}", lowBound=0, cat='Continuous') for i in range(n)]

# Restricao 1: cada cidade tem exatamente um sucessor
for i in range(n):
    prob += pulp.lpSum(x[i][j] for j in range(n) if j != i) == 1

# Restricao 2: cada cidade tem exatamente um predecessor
for j in range(n):
    prob += pulp.lpSum(x[i][j] for i in range(n) if i != j) == 1

# Restricao 3: eliminacao de sub-ciclos (MTZ)
for i in range(1, n):
    for j in range(1, n):
        if i != j:
            prob += u[i] - u[j] + n * x[i][j] <= n - 1

# Fixar u0 = 0 (opcional)
prob += u[0] == 0

# Funcao obj: min distancia total
prob += pulp.lpSum(dist[i][j] * x[i][j] for i in range(n) for j in range(n) if i != j)

prob.solve(pulp.PULP_CBC_CMD(msg=False))

# RESULTADOS (tabelas)
status = pulp.LpStatus[prob.status]
custo_total = pulp.value(prob.objective)

# Reconstruir rota (começando pela cidade 0)
edges_used = []
for i in range(n):
    for j in range(n):
        if i != j and x[i][j].varValue > 0.5:
            edges_used.append((i, j))

ordem = [0]
atual = 0
while len(ordem) < n:
    for j in range(n):
        if (atual, j) in edges_used:
            ordem.append(j)
            atual = j
            break
ordem.append(0)  # retorna p/ origem

nomes = [cidades[i] for i in ordem]

print("\nPROBLEMA DO CAIXEIRO VIAJANTE (TSP)\n")
print(f"Status: {status}")
print(f"Custo total mínimo: {custo_total:.0f}\n")

# Tabela da ordem de visita
df_rota = pd.DataFrame({
    'Passo': list(range(1, n+1)) + ['Retorno'],
    'Cidade': nomes
})
print("Rota ótima:")
print(df_rota.to_string(index=False))

# Tabela dos arcos percorridos
arcos = []
for k in range(len(ordem)-1):
    i, j = ordem[k], ordem[k+1]
    arcos.append({'Origem': cidades[i], 'Destino': cidades[j], 'Distancia': dist[i][j]})
df_arcos = pd.DataFrame(arcos)
print("\nArcos percorridos:")
print(df_arcos.to_string(index=False))